### **Mounting Drive**

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


### **Inputing the Requirement**

In [2]:
import datetime as dt, os

# filter config
TIMESTAMP     = dt.datetime.now().strftime('%d%m%Y')
COLUMNS = {
    'Date':      'Date',        # e.g. 'creation_time'
    'Media':     'Media',       # e.g. 'specific_resource_type'
    'Author':    'Author',      # e.g. 'from_username'
    'Mentions':  'Mentions',    # e.g. 'content'
    'Sentiment': 'Sentiment',   # e.g. 'final_sentiment'
    'Link':      'Link',
}
REPORT_TYPE     = 'Monthly' # 'Monthly' or 'Weekly' ('Daily' too for Environment)
ROBJ            = 'Indonesia' # Changes this based on your research object
ISSUES          = 'Environment' # 'Environment' or 'Politics'
PATH            = '/content/drive/MyDrive/Data/test data' # Path to access the data in google drive
FILE_NAME       = 'test_data_mingguan_bulanan.xlsx'   # one file, or a pattern: '*.xlsx', 'dpr26_*.csv', '*'
KEYWORDS        = []   # [] or None -> no filtering
REQUIRED        = []       # must-have terms (ALL of them); [] -> none
MIN_HITS        = 2                 # how many of KEYWORDS a row must match
EXACT_WORD      = False
DOMINANCE_RATIO = 2.0

### **Notebook Set Up**

In [3]:
%pip install -q "great[all] @ git+https://github.com/azmkto/GREAT-Tools.git"

import pandas as pd

# Everything that used to be a copy-pasted config cell:
from great import MEDIA_MAP, plat_class, sent_class, plat_colors, sent_colors, classify_issue, resolve_frame, validate_export
from great.viz import checks, prep
from great.viz.overview import monthly_overview, weekly_overview
from great.viz.wordcloud import sentiment_wordclouds
from great.viz.environment import dominant_issue, env_weekly, env_monthly, province_counts, island_counts
from great.viz.style import apply_style

apply_style()

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 24.5 MB/s eta 0:00:00


### **Accessing File**

In [4]:
df = pd.read_excel(f'{PATH}/{FILE_NAME}',header=1)
df1, start_date, end_date = prep.prepare_data(df)

sent_data = prep.sentiment_data(df1)    # Date x Sentiment (wide)
plat_data = prep.platform_data(df1)     # (Date, Media) -> Count
ps_data   = prep.plat_sent_data(df1)    # Media, Sentiment, Count (long)

print(f'{start_date} - {end_date}')

14 Sep 2026 - 20 Sep 2026


### **Checking Dataframe Info**

In [5]:
checks.frame_info(df1, 'Main Dataframe')


Main Dataframe Info:

Number of Rows: 19,000
Columns: ['No', 'Type', 'Headline', 'Mentions', 'Date', 'Link', 'Media', 'Sentiment', 'Author', 'Followers', 'Retweeted', 'Favourited', 'Location']

Data Types:
No                     int64
Type                  object
Headline              object
Mentions              object
Date          datetime64[ns]
Link                  object
Media                 object
Sentiment             object
Author                object
Followers              int64
Retweeted              int64
Favourited             int64
Location              object 



In [6]:
checks.frame_info(sent_data, 'Sentiment Dataframe')


Sentiment Dataframe Info:

Number of Rows: 7
Columns: ['negative', 'neutral', 'positive']

Data Types:
Sentiment
negative    int64
neutral     int64
positive    int64 



In [7]:
checks.frame_info(plat_data, 'Platform Dataframe')


Platform Dataframe Info:

Number of Rows: 42
Value Name: Count
Index Levels: ['Date', 'Media']
Data Type: int64
Date: 7 unique (14 Sep 2026 - 20 Sep 2026)
Media: 6 unique ['Facebook', 'Instagram', 'News', 'Tiktok', 'Twitter', 'Youtube']



In [8]:
checks.frame_info(ps_data, 'Platform-Sentiment Dataframe')


Platform-Sentiment Dataframe Info:

Number of Rows: 18
Columns: ['Media', 'Sentiment', 'Count']

Data Types:
Media        object
Sentiment    object
Count         int64 



In [9]:
checks.completeness(df1)

Completeness:
Headline     7754
Mentions       29
Author        185
Location    15601

Duplicate Rows     : 0
Duplicate Mentions : 9,594



In [10]:
checks.coverage(df1)

Time Coverage:
Days Covered  : 7 of 7
Empty Days    : 0
Peak Day      : 20 Sep 2026 (4,091 mentions)
Quietest Day  : 14 Sep 2026 (2,108 mentions)
Daily Average : 2,714 mentions



In [11]:
checks.composition(df1)

Sentiment Breakdown:
           Count  Share (%)
Sentiment                  
negative    7543       39.7
positive    6974       36.7
neutral     4483       23.6 

Platform Breakdown:
           Count  Share (%)
Media                      
News       10405       54.8
Twitter     7377       38.8
Youtube      841        4.4
Instagram    218        1.1
Facebook     121        0.6
Tiktok        38        0.2 



In [12]:
checks.author(df1)

Unique Authors: 8,654

Top 10 Authors:
                          Count  Share (%)
Author                                    
www.antaranews.com          253        1.3
www.metrotvnews.com         204        1.1
www.detik.com               199        1.0
matratni.sigapnews.co.id    162        0.9
mediaindonesia.com          153        0.8
www.liputan6.com            142        0.7
www.cnnindonesia.com        128        0.7
www.kompasiana.com          121        0.6
kalteng.antaranews.com      107        0.6
news.detik.com              105        0.6 



In [13]:
checks.labels(df1)

Label Check:
All Sentiment and Media values recognised


### **Visualize**

In [14]:
if ISSUES == 'Politics':
    if REPORT_TYPE == 'Weekly':
        sent_pct_df, platform_totals_df = weekly_overview(
            ps_data=ps_data,
            platform_data=plat_data,
            sentiment_data=sent_data,
            robj=ROBJ,
            start_date=start_date,
            end_date=end_date,
        )
    else:
        sent_pct_df, platform_totals_df = monthly_overview(
            ps_data=ps_data,
            platform_data=plat_data,
            sentiment_data=sent_data,
            robj=ROBJ,
            start_date=start_date,
            end_date=end_date,
            interval=3,
        )
else:
    df1['Isu_Inti'] = df1['Mentions'].map(classify_issue)
    df1[['Provinsi', 'Pulau']] = resolve_frame(df1)
    if REPORT_TYPE == 'Weekly':
        fig = env_weekly(df1, start_date=start_date, end_date=end_date,
                         dominance_ratio=2.0)   # top issue >= 2x the second -> it becomes a pie
    else:
        fig = env_monthly(df1, start_date=start_date, end_date=end_date, dominance_ratio=2.0)

In [16]:
if ISSUES == 'Politics':
    if REPORT_TYPE == 'Monthly':
        fig, terms_by_sentiment = sentiment_wordclouds(
            df1,
            sentiments=('positive', 'negative'),
            sent_colors=sent_colors,     # tints each cloud with its sentiment color
            exclude_media=('News',),     # pass () to keep news copy in
            n=200,                       # terms scored per sentiment
            max_words=100,               # terms actually drawn
        )
    else:
        pass
else:
    pass